<h1>Downloading Sentinel-3 OLCI Products</h1>

<p>Using the CDSE OData REST API to download Sentinel-3 chlorophyll-a products<br>
Level 2 Full Resolution Ocean Colour, Water, and Atmosphere Parameters<br>
Non Time Critical<br>
https://sentiwiki.copernicus.eu/web/olci-products for naming convention details<br>
<br>Note: The CHL_OC4ME and CHL_NN products are log10 scaled. Source: https://user.eumetsat.int/catalogue/EO:EUM:DAT:0027/overview<br>
<br>Documentation: https://documentation.dataspace.copernicus.eu/notebook-samples/geo/odata_basics.html</p>

In [8]:
import requests
import os
import zipfile
import re
import getpass

In [9]:
# function to get access token
def get_access_token(username, password):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    payload = {
        "grant_type": "password",
        "username": username,
        "password": password,
        "client_id": "cdse-public"
    }
    response = requests.post(url, data=payload)
    response.raise_for_status()
    return response.json()["access_token"]

# function for a sensing duration filter
# keep products with a sensing duration of 60 seconds or more
# ensures downloading of granules with meaningful spatial coverage over the AOI
def get_duration(product_name):
    # file name structure is always:
    # ..._{sensing_start}_{sensing_stop}_{processing_date}_{duration}_...
    # so find the 3 timestamps and take the segment right after them for duration
    parts = product_name.split("_")
    timestamp_pattern = re.compile(r"\d{8}T\d{6}")
    timestamp_indices = [i for i, p in enumerate(parts) if timestamp_pattern.fullmatch(p)]

    if len(timestamp_indices) >= 3:
        duration_index = timestamp_indices[2] + 1  # segment immediately after the 3rd timestamp
        return int(parts[duration_index])
    
    print(f"Could not parse duration from {product_name}, defaulting to 0")
    return 0

# timestamp 1 — Sensing start 20220101T132103 - when the satellite started collecting data for this granule e.g. January 1 2022 at 13:21:03 UTC
# timestamp 2 — Sensing stop 20220101T132134 - when the satellite stopped collecting data e.g. January 1 2022 at 13:21:34 UTC. Difference is 31 seconds.
# timestamp 3 — Processing date 20220103T061350 - when the ground station processed and generated the product e.g. January 3 2022 at 06:13:50 UTC, two days after data collection. This delay is normal for NTC (Non-Time-Critical) products, which go through more thorough processing than the near-real-time products.

# function to search for products
def search_sentinel3_water(start_date, end_date, bbox, min_duration=60):
    url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    all_products = []
    filtered_out = 0
    skip = 0
    page_size = 20

    while True:
        params = {
            "$filter": (
                f"Collection/Name eq 'SENTINEL-3' and "
                f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
                f"and att/OData.CSC.StringAttribute/Value eq 'OL_2_WFR___') and "
                f"ContentDate/Start gt {start_date}T00:00:00.000Z and "
                f"ContentDate/Start lt {end_date}T00:00:00.000Z and "
                f"OData.CSC.Intersects(area=geography'SRID=4326;POLYGON(({bbox}))')"
            ),
            "$top": page_size,
            "$skip": skip,
            "$orderby": "ContentDate/Start asc",
            "$count": "true"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        products = data["value"]
        total = data.get("@odata.count", "unknown")

        if skip == 0:
            print(f"Total products available (before duration filter): {total}")

        # apply duration filter
        for p in products:
            if get_duration(p["Name"]) >= min_duration:
                all_products.append(p)
            else:
                filtered_out += 1
                print(f"Skipping short fragment ({get_duration(p['Name'])}s): {p['Name']}")

        print(f"Fetched {skip + len(products)} / {total} products, kept {len(all_products)} so far.")

        if len(products) < page_size:
            break

        skip += page_size

    print(f"\nDuration filter removed {filtered_out} short fragment(s).")
    return all_products

# function to download the products
def download_product(product_id, product_name, token):
    downloads_folder = os.path.join(os.path.expanduser("~"), "Downloads")
    os.makedirs(downloads_folder, exist_ok=True)

    url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    headers = {"Authorization": f"Bearer {token}"}

    zip_path = os.path.join(downloads_folder, f"{product_name}.zip")
    sen3_path = os.path.join(downloads_folder, product_name)

    # define the specific files to keep
    # all available variables in OL_2_WFR:
    # variables_to_keep = [
    # "chl_oc4me.nc",   # Chlorophyll (OC4Me algorithm)
    # "chl_nn.nc",      # Chlorophyll (neural net)
    # "par.nc",         # Photosynthetically active radiation
    # "tsm_nn.nc",      # Total suspended matter
    # "iop_nn.nc",      # Inherent optical properties
    # "trsp.nc",        # Transparency
    # "w_aer.nc",       # Aerosol
    # "wqsf.nc",        # Water quality flags]

    variables_to_keep = [
    # science variables
    "chl_oc4me.nc",
    "chl_nn.nc",
    # mandatory files
    "geo_coordinates.nc",       # lat/lon for every pixel
    "tie_geo_coordinates.nc",   # lower resolution lat/lon grid
    "tie_geometries.nc",        # sun/view angles - needed for corrections
    "tie_meteo.nc",             # wind speed, pressure, humidity
    "instrument_data.nc",       # detector wavelengths and solar flux
    "time_coordinates.nc",      # per-scan timestamps
    "wqsf.nc",                  # water quality flags - tells us which pixels are valid
    "manifest.xml",             # product manifest
    "xfdumanifest.xml",         # full metadata
    "EOPMetadata.xml",          # Earth Observation metadata
]

    # skip if product was already downloaded and extracted
    if os.path.exists(sen3_path):
        print(f"Already exists, skipping: {product_name}")
        return

    # download
    print(f"Downloading: {product_name}")
    with requests.get(url, headers=headers, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Downloaded to: {zip_path}")

    # extract only specified files
    with zipfile.ZipFile(zip_path, "r") as z:
        all_files = z.namelist()
        target_files = [
            f for f in all_files
            if any(f.endswith(var) for var in variables_to_keep)
            or f.endswith("xfdumanifest.xml")
        ]

        if not target_files:
            print(f"None of {variables_to_keep} found in {product_name}")
        else:
            z.extractall(downloads_folder, members=target_files)
            print(f"Extracted: {[os.path.basename(f) for f in target_files]}")

    # delete the zip folder for cleanliness
    os.remove(zip_path)
    print(f"Removed zip: {zip_path}")

In [ ]:
# MAIN CODE

USERNAME = "gwynethcheng15@gmail.com"
PASSWORD = getpass.getpass("Enter your CDSE password: ")

token = get_access_token(USERNAME, PASSWORD)

# returns any granule that intersects the bounding box, but the downloaded .SEN3 file contains the full orbital strip
# format: bottom-left -> bottom-right -> top-right -> top-left -> close
# mcmurdo sound
bbox = "162.487793 -78.030132, 166.981201 -78.030132, 166.981201 -77.088607, 162.487793 -77.088607, 162.487793 -78.030132"

# date ranges:
# October 2022 - March 2023
# October 2023 - March 2024
# October 2024 - March 2025
# October 2025 - March 2026
products = search_sentinel3_water("2022-10-01", "2022-11-01", bbox) # last date exclusive
print(f"\nFound {len(products)} total products.")

# confirmation before downloading
# checkpoint since number of downloads can be large
confirm = input("\nProceed with download? (y/n): ")
if confirm.lower() == "y":
    for i, p in enumerate(products, start=1):
        print(f"\n[{i}/{len(products)}]")

        # refresh token every 2 products to avoid expiry mid-download
        # CDSE tokens typically expire after 10 minutes
        if i % 2 == 1:
            token = get_access_token(USERNAME, PASSWORD)
            print("Token refreshed.")

        download_product(p["Id"], p["Name"], token)

    print("\nAll downloads complete.")
else:
    print("Download cancelled.")

Total products available (before duration filter): 247
Skipping short fragment (55s): S3A_OL_2_WFR____20221101T165841_20221101T165937_20221103T093830_0055_091_311_4500_MAR_O_NT_003.SEN3
Fetched 20 / 247 products, kept 19 so far.
Fetched 40 / 247 products, kept 39 so far.
Fetched 60 / 247 products, kept 59 so far.
Fetched 80 / 247 products, kept 79 so far.
Fetched 100 / 247 products, kept 99 so far.
Fetched 120 / 247 products, kept 119 so far.
Fetched 140 / 247 products, kept 139 so far.
Fetched 160 / 247 products, kept 159 so far.
Fetched 180 / 247 products, kept 179 so far.
Fetched 200 / 247 products, kept 199 so far.
Fetched 220 / 247 products, kept 219 so far.
Skipping short fragment (5s): S3B_OL_2_WFR____20221129T141524_20221129T141530_20221201T072950_0005_073_181_4680_MAR_O_NT_003.SEN3
Skipping short fragment (9s): S3B_OL_2_WFR____20221130T134914_20221130T134923_20221202T070350_0009_073_195_4680_MAR_O_NT_003.SEN3
Skipping short fragment (9s): S3A_OL_2_WFR____20221130T142817_202211

<h1>Processing Downloaded OLCI Products</h1>

<p>Workflow:<br>
Step 1: Check grids of geo_coordinates.nc and science variables match, then attach lat/lon to the variables to get a swath NetCDF output<br>
Step 2: Apply WQSF quality masks, get a separate masked swath NetCDF output<br>
Step 3: Extract AOI pixels using lat/lon bounds<br>
Step 4: Compute daily/weekly statistics (likely to change)<br>
Step 5 (optional): Resample masked swath NetCDF outputs onto a regular grid, get GeoTIFF outputs in EPSG:5479 (MSLC2000) for QGIS visualisation</p>

In [ ]:
import netCDF4 as nc
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pyresample import geometry, kd_tree
from pyproj import CRS, Transformer
from datetime import datetime
from collections import defaultdict
import json

In [ ]:
# science variables to process
science_vars = {
    "chl_oc4me.nc": "CHL_OC4ME",
    "chl_nn.nc": "CHL_NN",
}

# recommended water quality flag combinations per science variable
# source: Table 1 of the pdf below
# https://user.eumetsat.int/s3/eup-strapi-media/S3_PN_OLCI_L2_M_003_05_Sentinel_3_Product_Notice_OLCI_Level_2_Ocean_Colour_21e0333d3d.pdf

# common flags for all three variables (under BAC and AAC):
# pixel must be (WATER or INLAND_WATER)
# and not any of (CLOUD, CLOUD_AMBIGUOUS, CLOUD_MARGIN, INVALID, COSMETIC, SATURATED, SUSPECT, HISOLZEN, HIGHGLINT, and SNOW_ICE)

# BAC processing chain flags:
# not any of (AC_FAIL, WHITECAPS, ADJAC, RWNEG_O2, RWNEG_O3, RWNEG_O4, RWNEG_O5, RWNEG_O6, RWNEG_O7, and RWNEG_O8)

# additional flags for each variable:
# CHL_OC4ME (BAC): also exclude OC4ME_FAIL
# CHL_NN (AAC): also exclude OCNN_FAIL

water_flags = ["WATER", "INLAND_WATER"]   # valid pixels

common_invalid_flags = [
    "CLOUD", "CLOUD_AMBIGUOUS", "CLOUD_MARGIN",
    "INVALID", "COSMETIC", "SATURATED", "SUSPECT",
    "HISOLZEN", "HIGHGLINT", "SNOW_ICE",
]

# BAC processing chain flags — apply only to CHL_OC4ME and PAR
BAC_chain_flags = [
    "AC_FAIL", "WHITECAPS", "ADJAC",
    "RWNEG_O2", "RWNEG_O3", "RWNEG_O4",
    "RWNEG_O5", "RWNEG_O6", "RWNEG_O7", "RWNEG_O8",
]

# flags per science variable + additional flags (invalid pixels)
variable_flags = {
    "CHL_OC4ME": BAC_chain_flags + ["OC4ME_FAIL"], # BAC product
    "CHL_NN": ["OCNN_FAIL"], # AAC product - no BAC chain flags needed
}

# bounding box
# mcmurdo sound
aoi_lat_min = -78.030132
aoi_lat_max = -77.088607
aoi_lon_min = 162.487793
aoi_lon_max = 166.981201

# target CRS and resolution for optional GeoTIFF export
target_crs = "EPSG:5479"
target_res_metres = 300

# STEP 1
# read science variables and metadata files in a .SEN3 folder
# check if grids in geo_coordinates.nc files match the grids of science variable files
# if they match, attach lat/lon in geo_coordinates.nc to science variables, get NetCDF files with embedded lat/lon as output
# swath_output_folder exports are optional
def check_and_write_swath_netcdf(sen3_folder, swath_output_folder=None):
    
    geo_path = os.path.join(sen3_folder, "geo_coordinates.nc")

    if not os.path.exists(geo_path):
        print(f"geo_coordinates.nc not found in {sen3_folder}")
        return None, None

    with nc.Dataset(geo_path) as geo:
        lat = geo.variables["latitude"][:] # shape: (rows, cols)
        lon = geo.variables["longitude"][:] # shape: (rows, cols)

    granule_name = os.path.basename(sen3_folder)
    print(f"\nGranule: {granule_name}")
    print(f"Coordinate grid shape: {lat.shape}")

    # check all grids match before writing anything
    data_arrays = {}
    all_ok = True

    for filename, varname in science_vars.items():
        var_path = os.path.join(sen3_folder, filename)

        if not os.path.exists(var_path):
            print(f"Missing: {filename}")
            all_ok = False
            continue

        with nc.Dataset(var_path) as ds:
            if varname not in ds.variables:
                print(f"'{varname}' not found in {filename}")
                all_ok = False
                continue
            data = ds.variables[varname][:]

        if data.shape != lat.shape:
            print(f"{filename} shape {data.shape} != coord shape {lat.shape}. Coordinate grids do not match.")
            all_ok = False
        else:
            print(f"{filename}: shape {data.shape} == coord shape {lat.shape}. Coordinate grids match!")
            data_arrays[varname] = data

    if not all_ok:
        print(f"Skipping swath NetCDF writing due to errors above.")
        return None, None

    # if swath_output_folder is set to None, outputs will be stored in in-memory path
    if swath_output_folder is None:
        print("swath_output_folder not set — keeping swath data in memory.")
        return None, {"lat": lat, "lon": lon, "arrays": data_arrays, "granule_name": granule_name}

    # if swath_output_folder is provided, export unmasked NetCDF with embedded lat/lon
    os.makedirs(swath_output_folder, exist_ok=True)
    out_path = os.path.join(swath_output_folder, f"{granule_name}_coordsembedded.nc")

    if os.path.exists(out_path):
        print(f"Swath NetCDF already exists, skipping: {out_path}")
        return out_path

    rows, cols = lat.shape

    with nc.Dataset(out_path, "w", format="NETCDF4") as ds:
        # global attributes
        ds.title = f"Sentinel-3 WFR swath (raw) — {granule_name}"
        ds.source = "Sentinel-3 OLCI Level-2 WFR, Copernicus Data Space Ecosystem"
        ds.Conventions = "CF-1.8"
        ds.crs = "WGS84 geographic (EPSG:4326) — swath coordinates"
        ds.quality_masking = "None - this is the raw unmasked file"

        # dimensions
        ds.createDimension("rows", rows)
        ds.createDimension("cols", cols)

        # latitude
        lat_var = ds.createVariable("latitude",  "f4", ("rows", "cols"), zlib=True)
        lat_var.units = "degrees_north"
        lat_var.long_name = "pixel latitude"
        lat_var.standard_name = "latitude"
        lat_var[:] = lat.data if np.ma.is_masked(lat) else lat

        # longitude
        lon_var = ds.createVariable("longitude", "f4", ("rows", "cols"), zlib=True)
        lon_var.units = "degrees_east"
        lon_var.long_name = "pixel longitude"
        lon_var.standard_name = "longitude"
        lon_var[:] = lon.data if np.ma.is_masked(lon) else lon

        # science variables
        for varname, data in data_arrays.items():
            v = ds.createVariable(
                varname, "f4", ("rows", "cols"),
                zlib=True, fill_value=np.float32(np.nan)
            )
            v.coordinates = "latitude longitude"
            v.grid_mapping = "crs"

            if varname == "CHL_OC4ME":
                v.long_name = "Chlorophyll-a concentration (OC4Me algorithm)"
                v.units     = "mg m-3"
            elif varname == "CHL_NN":
                v.long_name = "Chlorophyll-a concentration (neural network)"
                v.units     = "mg m-3"
            elif varname == "PAR":
                v.long_name = "Photosynthetically active radiation"
                v.units     = "microeinstein m-2 s-1"

            if np.ma.is_masked(data):
                v[:] = data.filled(np.nan).astype(np.float32)
            else:
                v[:] = data.astype(np.float32)

    print(f"Unmasked NetCDF with embedded coordinates written: {out_path}")
    return out_path, None

# ─────────────────────────────────────────────────────────────────────────────────────────
# STEP 2
# Apply WQSF quality masks, write separate masked NetCDF outputs

# function reads the bitmask value for a named flag from the WQSF variable metadata
# avoids hard-coding bitmask values, which can change between Sentinel-3 collection versions
def get_flag_bit(wqsf_dataset, flag_name):
    
    wqsf_var = wqsf_dataset.variables["WQSF"]
    flag_masks = wqsf_var.flag_masks # stores the full integer bitmask value for each flag (e.g. 1, 2, 4, 8, 65536 ...)
    flag_meanings = wqsf_var.flag_meanings.split()

    if flag_name not in flag_meanings:
        raise ValueError(
            f"Flag '{flag_name}' not found in WQSF."
            f"Available: {flag_meanings}"
        )
    return int(flag_masks[flag_meanings.index(flag_name)])


# function builds a boolean valid-pixel mask for a given science variable using the recommended flag combinations
# for each pixel: valid = WATER or INLAND_WATER
#                 AND NOT any common invalid flag
#                 AND NOT variable-specific fail flag
# returns a boolean array: True = include pixel, False = exclude pixel
def build_quality_mask(wqsf_ds, varname):

    wqsf = wqsf_ds.variables["WQSF"][:]

    # pixel must be classified as water
    water_mask = np.zeros(wqsf.shape, dtype=bool)
    for flag_name in water_flags:
        try:
            bit = get_flag_bit(wqsf_ds, flag_name)
            water_mask |= (wqsf & bit) != 0
        except ValueError as e:
            print(f"Warning: {e}")

    # pixel must not have any common invalid flag set
    invalid_mask = np.zeros(wqsf.shape, dtype=bool)
    for flag_name in common_invalid_flags:
        try:
            bit = get_flag_bit(wqsf_ds, flag_name)
            invalid_mask |= (wqsf & bit) != 0
        except ValueError as e:
            print(f"Warning: {e}")

    # pixel must not have the variable-specific flag set
    fail_mask = np.zeros(wqsf.shape, dtype=bool)
    for flag_name in variable_flags.get(varname, []):
        try:
            bit = get_flag_bit(wqsf_ds, flag_name)
            fail_mask |= (wqsf & bit) != 0
        except ValueError as e:
            print(f"Warning: {e}")

    valid = water_mask & ~invalid_mask & ~fail_mask
    return valid

# function reads wqsf.nc from the original .SEN3 folder and applies the recommended flag combinations to the raw NetCDF with embedded coordinates
# writes a separate masked NetCDF - the raw file is never modified
# accepts either a path (raw_swath_path) or an in-memory dict (swath_data) from check_and_write_swath_netcdf
# returns the path to the masked NetCDF output
def apply_quality_masks(sen3_folder, masked_output_folder,
                        raw_swath_path=None, swath_data=None):

    wqsf_path = os.path.join(sen3_folder, "wqsf.nc")
    if not os.path.exists(wqsf_path):
        print(f"wqsf.nc not found in {sen3_folder}, skipping masking.")
        return None

    granule_name = swath_data["granule_name"] if swath_data else os.path.basename(sen3_folder)

    os.makedirs(masked_output_folder, exist_ok=True)
    out_path = os.path.join(masked_output_folder, f"{granule_name}_qualitymasked.nc")

    if os.path.exists(out_path):
        print(f"Masked NetCDF already exists, skipping: {out_path}")
        return out_path

    print(f"Applying WQSF quality masks!")

    # build attribute lookup once, regardless of source
    # if coming from disk, read from the raw NetCDF
    # if in-memory, the attributes were never written so set them from the lookup table
    static_attrs = {
        "CHL_OC4ME": {"long_name": "Chlorophyll-a concentration (OC4Me algorithm)", "units": "mg m-3"},
        "CHL_NN": {"long_name": "Chlorophyll-a concentration (neural network)", "units": "mg m-3"},
        "PAR": {"long_name": "Photosynthetically active radiation", "units": "microeinstein m-2 s-1"},
    }

    def get_var_attrs(varname, raw_ds=None):
        if raw_ds is not None:
            return {attr: getattr(raw_ds.variables[varname], attr)
                    for attr in raw_ds.variables[varname].ncattrs()
                    if attr != "_FillValue"}
        return static_attrs.get(varname, {})

    # load lat/lon and data arrays from whichever source is available
    if raw_swath_path:
        with nc.Dataset(raw_swath_path) as raw_ds:
            lat = raw_ds.variables["latitude"][:]
            lon = raw_ds.variables["longitude"][:]
            varnames = [v for v in raw_ds.variables if v not in ("latitude", "longitude", "crs")]
            data_by_var = {v: raw_ds.variables[v][:].astype(np.float32) for v in varnames}
            var_attrs   = {v: get_var_attrs(v, raw_ds) for v in varnames}
    else:
        lat = swath_data["lat"]
        lon = swath_data["lon"]
        varnames = list(swath_data["arrays"].keys())
        data_by_var = {v: swath_data["arrays"][v].astype(np.float32) for v in varnames}
        var_attrs   = {v: get_var_attrs(v) for v in varnames}

    rows, cols = lat.shape

    with nc.Dataset(wqsf_path) as wqsf_ds, \
         nc.Dataset(out_path, "w", format="NETCDF4") as out_ds:

        out_ds.title = f"Sentinel-3 WFR swath (quality masked) — {granule_name}"
        out_ds.source = "Sentinel-3 OLCI Level-2 WFR, Copernicus Data Space Ecosystem"
        out_ds.Conventions = "CF-1.8"
        out_ds.crs = "WGS84 geographic (EPSG:4326) — swath coordinates"
        out_ds.quality_masking = (
            "EUMETSAT S3.PN-OLCI-L2M.003.05 Table 1: "
            "(WATER|INLAND_WATER) AND NOT (CLOUD|CLOUD_AMBIGUOUS|CLOUD_MARGIN|"
            "INVALID|COSMETIC|SATURATED|SUSPECT|HISOLZEN|HIGHGLINT|SNOW_ICE)"
            "AND NOT variable-specific fail flag"
        )

        out_ds.createDimension("rows", rows)
        out_ds.createDimension("cols", cols)

        lat_var = out_ds.createVariable("latitude", "f4", ("rows", "cols"), zlib=True)
        lat_var.units = "degrees_north"
        lat_var.long_name = "pixel latitude"
        lat_var.standard_name = "latitude"
        lat_var[:] = lat.data if np.ma.is_masked(lat) else lat

        lon_var = out_ds.createVariable("longitude", "f4", ("rows", "cols"), zlib=True)
        lon_var.units = "degrees_east"
        lon_var.long_name = "pixel longitude"
        lon_var.standard_name = "longitude"
        lon_var[:] = lon.data if np.ma.is_masked(lon) else lon

        for varname in varnames:
            data  = data_by_var[varname]
            valid = build_quality_mask(wqsf_ds, varname)

            masked_data = np.where(valid, data, np.nan).astype(np.float32)

            n_total  = valid.size
            n_valid  = int(np.sum(valid))
            n_invalid = n_total - n_valid
            pct_valid = n_valid / n_total * 100

            v = out_ds.createVariable(varname, "f4", ("rows", "cols"),
                                      zlib=True, fill_value=np.float32(np.nan))
            v.coordinates = "latitude longitude"
            v.grid_mapping = "crs"

            for attr, val in var_attrs[varname].items():
                setattr(v, attr, val)

            v[:] = masked_data

            print(f"{varname}: {n_valid}/{n_total} valid pixels "
                  f"({n_invalid} masked, {pct_valid:.1f}% valid)")

            if pct_valid < 10:
                print(f"Warning: only {pct_valid:.1f}% valid — possible sea ice / atmospheric correction issues.")

    print(f"Quality-masked NetCDF written: {out_path}")
    return out_path

# ─────────────────────────────────────────────────────────────────────────────────────────
# STEP 3
# Extract AOI pixels using lat/lon bounds

# function reads a quality-masked NetCDF and extracts all valid pixels that fall within the AOI bounding box, identified by their embedded lat/lon values
def extract_aoi_pixels(swath_nc_path,
                       lat_min=aoi_lat_min, lat_max=aoi_lat_max,
                       lon_min=aoi_lon_min, lon_max=aoi_lon_max):

    with nc.Dataset(swath_nc_path) as ds:
        lat = ds.variables["latitude"][:]
        lon = ds.variables["longitude"][:]

        aoi_mask = (
            (lat >= lat_min) & (lat <= lat_max) &
            (lon >= lon_min) & (lon <= lon_max)
        )

        n_aoi = int(np.sum(aoi_mask))
        if n_aoi == 0:
            print(f"No pixels found within AOI for {os.path.basename(swath_nc_path)}")
            return None

        print(f"AOI pixels found: {n_aoi}")

        results = {}
        varnames = [v for v in ds.variables if v not in ("latitude", "longitude", "crs")]

        for varname in varnames:
            data = ds.variables[varname][:].astype(np.float32)
            aoi_values = data[aoi_mask]
            valid_values = aoi_values[~np.isnan(aoi_values)]

            n_valid = len(valid_values)
            if n_valid == 0:
                print(f"{varname}: no valid pixels in AOI after quality masking")
            else:
                print(f"{varname}: {n_valid} valid pixels in AOI")

            results[varname] = valid_values  # empty array if none valid

    return results

# ─────────────────────────────────────────────────────────────────────────────────────────
# STEP 4
# Compute daily and weekly statistics (subject to change based on methodology)

# function parses the sensing start date from the raw swath NetCDF filename
# Sentinel-3 granule names follow the pattern: S3A_OL_2_WFR_____{YYYYMMDDTHHMMSS}_...
def parse_sensing_date(swath_nc_filename):

    basename = os.path.basename(swath_nc_filename)
    parts = basename.split("_")
    pattern = re.compile(r"\d{8}T\d{6}")
    timestamps = [p for p in parts if pattern.fullmatch(p)]

    if not timestamps:
        raise ValueError(f"Could not parse sensing date from {basename}")

    return datetime.strptime(timestamps[0], "%Y%m%dT%H%M%S").date()


# function iterates over all quality-masked NetCDFs, extracts AOI pixel values per granule
# then aggregates into daily and weekly statistics
# multiple granules on the same day (ascending/descending passes) are pooled together before computing daily statistics
def compute_statistics(masked_output_folder,
                       lat_min=aoi_lat_min, lat_max=aoi_lat_max,
                       lon_min=aoi_lon_min, lon_max=aoi_lon_max):
    
    masked_files = sorted([
        os.path.join(masked_output_folder, f)
        for f in os.listdir(masked_output_folder)
        if f.endswith("_qualitymasked.nc")
    ])

    if not masked_files:
        print("No quality-masked NetCDF files found.")
        return None, None

    print(f"\nFound {len(masked_files)} quality-masked NetCDF(s) to process.\n")

    # accumulate raw pixel values per day across all granules
    daily_values = defaultdict(lambda: defaultdict(list))

    for swath_path in masked_files:
        print(f"Extracting AOI pixels: {os.path.basename(swath_path)}")
        try:
            date = parse_sensing_date(swath_path)
        except ValueError as e:
            print(f"Warning: {e}")
            continue

        pixels = extract_aoi_pixels(swath_path, lat_min, lat_max, lon_min, lon_max)
        if pixels is None:
            continue
        
        # for each science variable, appends the extracted pixel values into the corresponding date bucket
        # .extend() rather than .append() flattens the array into the list
        # so if two granules on the same day each contribute 500 pixels, you end up with one flat list of 1000 values for that day, not two nested lists of 500
        for varname, values in pixels.items():
            if len(values) > 0:
                daily_values[date][varname].extend(values.tolist())

    # daily statistics
    # once all granules are processed, loops over each date and converts the pooled pixel list into a numpy array
    # the daily statistics are then computed directly from that flat list of raw individual pixel values — not from averages of averages
    # nanmean/nanmedian/nanpercentile are used as a safety measure, which ignore the NaN and computes from remaining values
    
    # for CHL_OC4ME and CHL_NN, use geometric mean:
    # instead of averaging the raw values, average the log-transformed values - makes it less sensitive to extreme high values and better represents the "typical" pixel in a skewed distribution (typical of chlorophyll values)
    daily_stats = {}
    for date, var_dict in sorted(daily_values.items()):
        daily_stats[date] = {}
        for varname, values in var_dict.items():
            arr = np.array(values, dtype=np.float32)
            daily_stats[date][varname] = {
                "mean": float(np.nanmean(arr)),
                "median": float(np.nanmedian(arr)),
                "p25": float(np.nanpercentile(arr, 25)),
                "p75": float(np.nanpercentile(arr, 75)),
                "n_pixels": len(arr), # how many valid pixels contributed to that day's statistics - useful for flagging days with sparse coverage
            }

    # aggregate daily means into ISO weekly means
    # note that weekly statistics are computed as the mean of daily means, not the mean of all raw pixels
    # this gives each day equal weight regardless of how many granules or pixels it had
    weekly_values = defaultdict(lambda: defaultdict(list))
    for date, var_dict in daily_stats.items():
        iso_week = f"{date.isocalendar()[0]}-W{date.isocalendar()[1]:02d}"
        for varname, s in var_dict.items():
            if not np.isnan(s["mean"]):
                weekly_values[iso_week][varname].append(s["mean"])

    weekly_stats = {}
    for iso_week, var_dict in sorted(weekly_values.items()):
        weekly_stats[iso_week] = {}
        for varname, means in var_dict.items():
            weekly_stats[iso_week][varname] = {
                "mean":   float(np.nanmean(means)),
                "n_days": len(means),
            }

    # print summary
    print("\nDaily statistics (McMurdo Sound)")
    for date, var_dict in sorted(daily_stats.items()):
        print(f"{date}")
        for varname, s in var_dict.items():
            print(f"{varname}: mean={s['mean']:.4f}, median={s['median']:.4f}, "
                  f"p25={s['p25']:.4f}, p75={s['p75']:.4f}, n={s['n_pixels']}")

    print("\nWeekly statistics (McMurdo Sound)")
    for iso_week, var_dict in sorted(weekly_stats.items()):
        print(f"{iso_week}")
        for varname, s in var_dict.items():
            print(f"{varname}: mean={s['mean']:.4f}, n_days={s['n_days']}")

    return daily_stats, weekly_stats

# function to save daily statistics in JSON files for visualisation
def save_daily_stats_json(daily_stats, output_folder):

    os.makedirs(output_folder, exist_ok=True)
    out_path = os.path.join(output_folder, "SEN3_daily_stats.json")

    # date keys are converted to strings since JSON doesn't support date objects
    serialisable = {str(k): v for k, v in daily_stats.items()}
    with open(out_path, "w") as f:
        json.dump(serialisable, f, indent=2)

    print(f"Daily stats saved: {out_path}")
    return out_path

# ─────────────────────────────────────────────────────────────────────────────────────────
# STEP 5 (optional)
# Resample masked swath NetCDF to compressed GeoTIFF outputs
# Reprojected to EPSG:5479 (RSRGD2000 / MSLC2000)

def resample_swath_to_geotiff(masked_swath_path, geotiff_output_folder,
                               target_crs=target_crs,
                               resolution_m=target_res_metres):

    if not os.path.exists(masked_swath_path):
        print(f"Error: Quality-masked NetCDF not found: {masked_swath_path}")
        return

    granule_name = os.path.splitext(os.path.basename(masked_swath_path))[0]
    os.makedirs(geotiff_output_folder, exist_ok=True)

    with nc.Dataset(masked_swath_path, "r") as ds:
        lat = ds.variables["latitude"][:]
        lon = ds.variables["longitude"][:]
        varnames = [v for v in ds.variables if v not in ("latitude", "longitude", "crs")]
        data_arrays = {v: ds.variables[v][:] for v in varnames}

    print(f"\nResampling: {granule_name}")

    # project NetCDF lat/lon into target CRS to define output grid extent
    transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    x_proj, y_proj = transformer.transform(lon.ravel(), lat.ravel())

    x_min, x_max = x_proj.min(), x_proj.max()
    y_min, y_max = y_proj.min(), y_proj.max()

    # define output grid dimensions from extent and resolution
    grid_cols = int(np.ceil((x_max - x_min) / resolution_m))
    grid_rows = int(np.ceil((y_max - y_min) / resolution_m))

    print(f"Output grid: {grid_rows} rows x {grid_cols} cols at {resolution_m}m")

    # build pyresample swath definition from original lat/lon
    swath_def = geometry.SwathDefinition(
        lons=lon.data if np.ma.is_masked(lon) else lon,
        lats=lat.data if np.ma.is_masked(lat) else lat
    )

    # build pyresample area definition for the target projected grid
    crs_obj  = CRS.from_epsg(int(target_crs.split(":")[1]))
    area_def = geometry.AreaDefinition(
        area_id = "mslc2000_grid",
        description = f"MSLC2000 {resolution_m}m grid",
        proj_id = target_crs,
        projection = crs_obj.to_dict(),
        width = grid_cols,
        height = grid_rows,
        area_extent = (x_min, y_min, x_max, y_max)
    )

    # resample each science variable using nearest-neighbour
    # radius_of_influence: search radius in metres for finding swath pixels
    # set to 2x the native resolution to avoid gaps without over-smoothing
    radius = resolution_m * 2

    # affine transform for rasterio (top-left origin, positive x, negative y)
    transform = from_bounds(x_min, y_min, x_max, y_max, grid_cols, grid_rows)

    for varname, data in data_arrays.items():
        out_filename = f"{granule_name}_{varname}.tif"
        out_path     = os.path.join(geotiff_output_folder, out_filename)

        if os.path.exists(out_path):
            print(f"Already exists, skipping: {out_filename}")
            continue

        # fill masked values before passing to pyresample
        if np.ma.is_masked(data):
            data_filled = data.filled(np.nan).astype(np.float32)
        else:
            data_filled = data.astype(np.float32)

        # nearest-neighbour resampling from swath to regular grid
        resampled = kd_tree.resample_nearest(
            swath_def, data_filled, area_def,
            radius_of_influence=radius,
            fill_value=np.nan
        )

        # write GeoTIFF
        with rasterio.open(
            out_path, "w",
            driver    = "GTiff",
            height    = grid_rows,
            width     = grid_cols,
            count     = 1,
            dtype     = np.float32,
            crs       = target_crs,
            transform = transform,
            nodata    = np.nan,
            compress  = "lzw",
        ) as dst:
            dst.write(resampled.astype(np.float32), 1)

        print(f"Exported GeoTIFF: {out_path}")

# ─────────────────────────────────────────────────────────────────────────────────────────
# FULL PIPELINE

# Run all .SEN3granules through the full pipeline:
# Step 1: derive NetCDF with embedded lat/lon in swath_output_folder
# Step 2: derive quality-masked NetCDF in masked_output_folder
# Step 3: extract AOI pixels from quality-masked NetCDFs
# Step 4: calculate statistics
# Step 5 (optional): resample NetCDFs to GeoTIFFs in geotiff_output_folder

def process_all_granules(downloads_folder, swath_output_folder,
                         masked_output_folder, stats_output_folder=None,
                         geotiff_output_folder=None):

    sen3_folders = sorted([
        os.path.join(downloads_folder, f)
        for f in os.listdir(downloads_folder)
        if f.endswith(".SEN3")
    ])

    if not sen3_folders:
        print("No .SEN3 folders found.")
        return None, None, None

    print(f"Found {len(sen3_folders)} granule(s).\n")

    for folder in sen3_folders:
        raw_swath_path, swath_data = check_and_write_swath_netcdf(folder, swath_output_folder)

        if raw_swath_path is None and swath_data is None:
            continue

        masked_path = apply_quality_masks(
            folder, masked_output_folder,
            raw_swath_path=raw_swath_path,
            swath_data=swath_data,
        )

        if geotiff_output_folder and masked_path:
            resample_swath_to_geotiff(masked_path, geotiff_output_folder)

    # Steps 3 and 4: extract AOI pixels and calculate statistics from all quality-masked files
    daily_stats, weekly_stats = compute_statistics(masked_output_folder)

    # save daily stats to JSON (optional)
    if stats_output_folder:
        save_daily_stats_json(daily_stats, stats_output_folder)

    return daily_stats, weekly_stats

In [ ]:
# MAIN CODE
# for swath_output and geotiff_output, set to None to skip exporting and keep outputs in memory

downloads_folder = ("/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_0_RAWDATA/Sentinel3/Oct2022_Mar2023")
swath_output_folder = None
masked_output_folder  = ("/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/Sentinel3/netcdf_wqsfmasked/Sentinel3")
stats_output_folder = ("/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/Sentinel3/jsons_dailystats")
geotiff_output_folder = ("/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_MASKING_SUMMARYSTATS/Sentinel3/geotiffs_resampled/SEN3TEST")

daily_stats, weekly_stats = process_all_granules(
    downloads_folder,
    swath_output_folder,
    masked_output_folder,
    stats_output_folder,
    geotiff_output_folder,
)

<h3>Notes (resampling to GeoTIFFs step):</h3>
<p>On radius_of_influence: Set to resolution_m * 2 (600m). Too small and we might get gaps in the output where swath pixels are sparse; too large and we might get over-smoothing. Tune this depending on how the granules look in QGIS.<br>
On nearest-neighbour vs other resamplers: This code uses nearest-neighbour (resample_nearest) which preserves original values without interpolation — good for chlorophyll data (averaging across water/land boundaries would produce artefacts). If the output looks patchy, kd_tree.resample_gauss is a good alternative that applies a Gaussian weighting over nearby pixels.</p>